# Packet P-039 — Conformal Calibration Under Temporal Shift

**Decision question:** Are P-017's conformal prediction intervals still calibrated when deployed forward in time (given P-035's temporal drift)?

**Fastest falsifier:** 50/20/30 temporal split (train/cal/test). If 80% coverage on newest 30% drops below 60%, conformal guarantees break under temporal shift.

**Success:** Temporal conformal 80% coverage ≥ 70%.
**Kill:** Coverage < 60% — conformal intervals are not deployment-safe under temporal shift.

In [1]:
"""Cell 1 — Load data, temporal split, conformal calibration."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

# Sort by Ref_ID (temporal proxy)
sort_order = df['Ref_ID'].argsort().values
X_sorted = X.iloc[sort_order].values
y_sorted = y.iloc[sort_order].values

n = len(df)
n_train = int(n * 0.50)
n_cal = int(n * 0.20)
n_test = n - n_train - n_cal

X_train = X_sorted[:n_train]
y_train = y_sorted[:n_train]
X_cal = X_sorted[n_train:n_train + n_cal]
y_cal = y_sorted[n_train:n_train + n_cal]
X_test = X_sorted[n_train + n_cal:]
y_test = y_sorted[n_train + n_cal:]

print(f"Train: {n_train} (oldest 50%)")
print(f"Calibration: {n_cal} (middle 20%)")
print(f"Test: {n_test} (newest 30%)")

# Train model
model = ExtraTreesRegressor(**ET_PARAMS)
model.fit(X_train, y_train)

# Per-tree predictions for uncertainty (needed for conformal)
tree_preds_cal = np.array([t.predict(X_cal) for t in model.estimators_])
tree_preds_test = np.array([t.predict(X_test) for t in model.estimators_])

pred_cal = tree_preds_cal.mean(axis=0)
pred_test = tree_preds_test.mean(axis=0)
std_cal = tree_preds_cal.std(axis=0)
std_test = tree_preds_test.std(axis=0)

# Conformity scores on calibration set
residuals_cal = np.abs(y_cal - pred_cal)

# Test multiple nominal coverage levels
coverages = [0.80, 0.90, 0.95]
results = []

print(f"\n{'=' * 60}")
print("CONFORMAL CALIBRATION UNDER TEMPORAL SHIFT")
print(f"{'=' * 60}")
print(f"{'Nominal':>8} {'q-hat':>8} {'Temporal':>10} {'Random*':>10}")
print("-" * 40)

for alpha in coverages:
    # q-hat from calibration set
    q_hat = np.quantile(residuals_cal, alpha)
    
    # Coverage on temporally distant test set
    residuals_test = np.abs(y_test - pred_test)
    temporal_coverage = np.mean(residuals_test <= q_hat)
    
    results.append({
        'nominal': alpha,
        'q_hat': q_hat,
        'temporal_coverage': temporal_coverage,
    })
    
    # P-017 random-split reference
    random_ref = {0.80: 0.799, 0.90: 0.899, 0.95: 0.951}
    ref = random_ref.get(alpha, 'N/A')
    
    print(f"{alpha:>8.0%} {q_hat:>8.3f} {temporal_coverage:>10.1%} {ref:>10}")

# Also check by T80 range
print(f"\n── Coverage by T80 range (80% nominal) ──")
q80 = results[0]['q_hat']
t80_test = np.expm1(y_test)
residuals_test = np.abs(y_test - pred_test)

ranges = [(0, 100), (100, 500), (500, 1000), (1000, 10000)]
for lo, hi in ranges:
    mask = (t80_test >= lo) & (t80_test < hi)
    if mask.sum() > 5:
        cov = np.mean(residuals_test[mask] <= q80)
        print(f"  T80 {lo}-{hi}h (n={mask.sum()}): {cov:.1%}")

# Tau-b on test set
tau_test, _ = kendalltau(y_test, pred_test)
print(f"\nTau-b on temporal test set: {tau_test:.4f} (vs 0.289 random baseline)")

Train: 771 (oldest 50%)
Calibration: 308 (middle 20%)
Test: 464 (newest 30%)



CONFORMAL CALIBRATION UNDER TEMPORAL SHIFT
 Nominal    q-hat   Temporal    Random*
----------------------------------------
     80%    2.308      77.2%      0.799
     90%    3.360      88.1%      0.899
     95%    3.887      91.6%      0.951

── Coverage by T80 range (80% nominal) ──
  T80 0-100h (n=202): 57.4%
  T80 100-500h (n=174): 99.4%
  T80 500-1000h (n=61): 88.5%
  T80 1000-10000h (n=27): 55.6%

Tau-b on temporal test set: 0.0990 (vs 0.289 random baseline)


In [2]:
"""Cell 2 — Evaluate and save."""

cov_80 = results[0]['temporal_coverage']

print("── Evaluation ──\n")
print(f"80% nominal temporal coverage: {cov_80:.1%}")
print(f"P-017 random-split 80% coverage: 79.9%")
print(f"Gap: {cov_80 - 0.799:+.1%}pp")

if cov_80 >= 0.70:
    status = "Confirmed"
elif cov_80 >= 0.60:
    status = "Promising"
else:
    status = "Negative"

# Save
pd.DataFrame(results).to_csv('Packet_P039_Conformal_Temporal.csv', index=False)
print(f"\nSaved: Packet_P039_Conformal_Temporal.csv")

print(f"\n{'=' * 60}")
print(f"P-039 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Conformal intervals remain reasonably calibrated under temporal shift.")
elif status == "Promising":
    print("Coverage degrades but stays above 60% — partial calibration.")
else:
    print("Conformal guarantees break under temporal shift (coverage < 60%).")
    print("Intervals are not deployment-safe without retraining.")

── Evaluation ──

80% nominal temporal coverage: 77.2%
P-017 random-split 80% coverage: 79.9%
Gap: -2.7%pp

Saved: Packet_P039_Conformal_Temporal.csv

P-039 STATUS: Confirmed
Conformal intervals remain reasonably calibrated under temporal shift.
